# Sampling-depth sensitivity: complete final QC versus nominal first 100 trajectories

Here we compare complete final-QC ensembles with a fixed nominal early cohort defined before QC: the first 20 ordered seeds × five AlphaFold model trajectories = 100 intended trajectories per protocol. Only trajectories surviving the same final-QC procedure are analyzed. This is a sampling-depth sensitivity analysis, not an independent replicate or a chronological record of model generation.

CaV1.2 and NaV1.5 use their final 3 Å allOK3 distance subsets. Kv2.1 uses the stricter allOK3 structural-interface/alignment-QC subset. The NaV1.5 comparison is QQQ vanilla versus the original mask v1, matching the primary mutant analysis. Experimental markers retain the same PDB-specific colors and shapes used in the channel notebooks.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

repo_root = Path.cwd() if (Path.cwd() / 'shared').is_dir() else Path.cwd().parent
sys.path.insert(0, str(repo_root)) if str(repo_root) not in sys.path else None

from shared.data_access import resolve

from shared.plotting import CAV12_PALETTE, KV21_PALETTE, NAV15_PALETTE, ACCENT_PALETTE
from shared.sampling_depth_analysis import (
    SUBSETS, load_final_qc_pair, plot_sampling_depth_condition,
    summarize_sampling_depth,
)
from shared.presentation_export import install_notebook_figure_export

install_notebook_figure_export(
    repo_root.parent / 'vgci_mutants_writing' / 'figures' / 'cross_project',
    'SamplingDepth_AllOK3_vs_First100',
)
sns.set_theme(style='whitegrid', context='notebook')


## Final-QC dataset catalog


In [ ]:
DATASETS = {
    'CaV1.2 WT': {
        'channel': 'Cav1.2', 'condition': 'WT',
        'colors': {'vanilla': CAV12_PALETTE['WT_VAN'], 'masked': CAV12_PALETTE['WT_HM']},
        'vanilla': resolve('cav12/dataDistances/26-02-10_Cav12_wt_vanillaAF2_distances_all_ok_rmsd_3A.csv'),
        'masked': resolve('cav12/dataDistances/26-02-10_Cav12_wt_maskedAF2_distances_all_ok_rmsd_3A.csv'),
    },
    'CaV1.2 G402S': {
        'channel': 'Cav1.2', 'condition': 'G402S',
        'colors': {'vanilla': CAV12_PALETTE['G402S_VAN'], 'masked': CAV12_PALETTE['G402S_HM']},
        'vanilla': resolve('cav12/dataDistances/26-02-10_Cav12_g402s_vanillaAF2_distances_all_ok_rmsd_3A.csv'),
        'masked': resolve('cav12/dataDistances/26-02-10_Cav12_g402s_maskedAF2_distances_all_ok_rmsd_3A.csv'),
    },
    'CaV1.2 G406R': {
        'channel': 'Cav1.2', 'condition': 'G406R',
        'colors': {'vanilla': CAV12_PALETTE['G406R_VAN'], 'masked': CAV12_PALETTE['G406R_HM']},
        'vanilla': resolve('cav12/dataDistances/26-02-10_Cav12_g406r_vanillaAF2_distances_all_ok_rmsd_3A.csv'),
        'masked': resolve('cav12/dataDistances/26-02-10_Cav12_g406r_maskedAF2_distances_all_ok_rmsd_3A.csv'),
    },
    'Kv2.1 WT': {
        'channel': 'Kv2.1', 'condition': 'WT',
        'colors': {'vanilla': KV21_PALETTE['WT_VAN'], 'masked': KV21_PALETTE['WT_HM']},
        'vanilla': resolve('kv21/dataDistances/26-02-11_Kv2.1_wt_vanillaAF2_distances_all_ok_rmsd_3A_structural_interface_alignment_qc.csv'),
        'masked': resolve('kv21/dataDistances/26-02-11_Kv2.1_wt_maskedAF2_distances_all_ok_rmsd_3A_structural_interface_alignment_qc.csv'),
    },
    'Kv2.1 L403A': {
        'channel': 'Kv2.1', 'condition': 'L403A',
        'colors': {'vanilla': KV21_PALETTE['L403A_VAN'], 'masked': KV21_PALETTE['L403A_HM']},
        'vanilla': resolve('kv21/dataDistances/26-02-11_Kv2.1_l403a_vanillaAF2_distances_all_ok_rmsd_3A_structural_interface_alignment_qc.csv'),
        'masked': resolve('kv21/dataDistances/26-02-11_Kv2.1_l403a_maskedAF2_distances_all_ok_rmsd_3A_structural_interface_alignment_qc.csv'),
    },
    'Kv2.1 F412L': {
        'channel': 'Kv2.1', 'condition': 'F412L',
        'colors': {'vanilla': KV21_PALETTE['F412L_VAN'], 'masked': KV21_PALETTE['F412L_HM']},
        'vanilla': resolve('kv21/dataDistances/26-02-11_Kv2.1_f412l_vanillaAF2_distances_all_ok_rmsd_3A_structural_interface_alignment_qc.csv'),
        'masked': resolve('kv21/dataDistances/26-02-11_Kv2.1_f412l_maskedAF2_distances_all_ok_rmsd_3A_structural_interface_alignment_qc.csv'),
    },
    'NaV1.5 QQQ': {
        'channel': 'Nav1.5', 'condition': 'QQQ | original mask v1',
        'colors': {'vanilla': NAV15_PALETTE['QQQ_VAN'], 'masked': NAV15_PALETTE['QQQ_HM']},
        'vanilla': resolve('nav15/dataDistances/26-07-27_Nav15_qqq_vanilla_AF2_distances_all_ok_rmsd_3A.csv'),
        'masked': resolve('nav15/dataDistances/26-07-27_Nav15_qqq_masked_AF2_distances_all_ok_rmsd_3A.csv'),
    },
}

path_audit = pd.DataFrame([
    {'Dataset': name, 'Protocol': protocol, 'Path': str(config[protocol]), 'Exists': config[protocol].is_file()}
    for name, config in DATASETS.items() for protocol in ('vanilla', 'masked')
])
display(path_audit)
assert path_audit['Exists'].all(), 'A required final-QC distance table is missing.'


## Structural coordinates

The comparison uses the same selectivity-filter, voltage-sensor, and intracellular-gate coordinates as the channel notebooks. Experimental overlays are included only where the selected aliases have validated residue correspondences. The Kv2.1 WT panels additionally use 9O10–9O13 as an experimental state ladder: VSD-up/high-K, VSD-up/low-K, two-VSD-down/open-pore, and four-VSD-down/constricted-pore. These state markers are omitted from the mutant panels because they are not mutation-matched references.


In [ ]:
REGIONS = {
    'Cav1.2': {
        'selectivity_filter': {'E363-E706':'CA_GLU363_CA-GLU706_CA', 'E363-E1135':'CA_GLU363_CA-GLU1135_CA', 'E363-E1464':'CA_GLU363_CA-GLU1464_CA', 'E706-E1135':'CA_GLU706_CA-GLU1135_CA', 'E706-E1464':'CA_GLU706_CA-GLU1464_CA', 'E1135-E1464':'CA_GLU1135_CA-GLU1464_CA'},
        'vsds': {'F170-R237':'CA_ARG237_CA-PHE170_CA', 'F170-R246':'CA_ARG246_CA-PHE170_CA', 'F567-R620':'CA_ARG620_CA-PHE567_CA', 'F567-K629':'CA_LYS629_CA-PHE567_CA', 'F940-K1015':'CA_LYS1015_CA-PHE940_CA', 'F940-K1031':'CA_ARG1031_CA-PHE940_CA', 'F1278-R1377':'CA_ARG1377_CA-PHE1278_CA', 'F1278-K1386':'CA_LYS1386_CA-PHE1278_CA'},
        'intracellular_gate': {'L401-L749':'CA_LEU401_CA-LEU749_CA', 'L401-V1182':'CA_LEU401_CA-VAL1182_CA', 'L401-I1516':'CA_LEU401_CA-ILE1516_CA', 'L749-V1182':'CA_LEU749_CA-VAL1182_CA', 'L749-I1516':'CA_LEU749_CA-ILE1516_CA', 'V1182-I1516':'CA_VAL1182_CA-ILE1516_CA'},
    },
    'Kv2.1': {
        'selectivity_filter': {'G375 A-B':'CA_CA_A_GLY377_CA-B_GLY377_CA', 'G375 A-C':'CA_CA_A_GLY377_CA-C_GLY377_CA', 'G375 A-D':'CA_CA_A_GLY377_CA-D_GLY377_CA', 'G375 B-C':'CA_CA_B_GLY377_CA-C_GLY377_CA', 'G375 B-D':'CA_CA_B_GLY377_CA-D_GLY377_CA', 'G375 C-D':'CA_CA_C_GLY377_CA-D_GLY377_CA'},
        'vsds': {'F236-R289 A':'CA_CA_A_ARG291_CA-A_PHE238_CA', 'F236-R308 A':'CA_CA_A_ARG310_CA-A_PHE238_CA'},
        'intracellular_gate': {'A402 A-B':'CA_CA_A_ALA404_CA-B_ALA404_CA', 'A402 A-C':'CA_CA_A_ALA404_CA-C_ALA404_CA', 'A402 A-D':'CA_CA_A_ALA404_CA-D_ALA404_CA', 'A402 B-C':'CA_CA_B_ALA404_CA-C_ALA404_CA', 'A402 B-D':'CA_CA_B_ALA404_CA-D_ALA404_CA', 'A402 C-D':'CA_CA_C_ALA404_CA-D_ALA404_CA'},
    },
    'Nav1.5': {
        'selectivity_filter': {'D373-E704':'CA_ASP373_CA-GLU704_CA', 'D373-K1103':'CA_ASP373_CA-LYS1103_CA', 'D373-A1395':'CA_ASP373_CA-ALA1395_CA', 'E704-K1103':'CA_GLU704_CA-LYS1103_CA', 'E704-A1395':'CA_GLU704_CA-ALA1395_CA', 'K1103-A1395':'CA_LYS1103_CA-ALA1395_CA'},
        'vsds': {'F165-R220':'CA_ARG220_CA-PHE165_CA', 'F165-R229':'CA_LYS229_CA-PHE165_CA', 'F560-R612':'CA_ARG612_CA-PHE560_CA', 'F560-K624':'CA_LYS624_CA-PHE560_CA', 'F934-R984':'CA_LYS984_CA-PHE934_CA', 'F934-K1000':'CA_ARG1000_CA-PHE934_CA', 'F1251-R1307':'CA_ARG1307_CA-PHE1251_CA', 'F1251-R1322':'CA_ARG1322_CA-PHE1251_CA'},
        'intracellular_gate': {'M415-A742':'CA_MET415_CA-ALA742_CA', 'M415-I1154':'CA_MET415_CA-ILE1154_CA', 'M415-I1455':'CA_MET415_CA-ILE1455_CA', 'A742-I1154':'CA_ALA742_CA-ILE1154_CA', 'A742-I1455':'CA_ALA742_CA-ILE1455_CA', 'I1154-I1455':'CA_ILE1154_CA-ILE1455_CA'},
    },
}


## Complete versus reduced sampling

Each trajectory contributes one structure: its latest recycle surviving final QC. Each upper panel uses these representatives from the complete final-QC cohort. Each lower panel uses representatives surviving from the nominal first 100 trajectories: five model trajectories from each of the first 20 ordered seeds. This keeps the intended seed/model cohort fixed before QC rather than extending farther into the run until 100 survivors are accumulated. A difference between panels indicates sensitivity to sampling depth; it does not establish a separate conformational population.

In [ ]:
summary_rows = []
ranking_tables = {}
figure_dir = repo_root / 'docs' / 'figures' / 'sampling_depth'
table_dir = repo_root / 'docs' / 'tables' / 'sampling_depth'
figure_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

for name, config in DATASETS.items():
    print(f'Loading {name}')
    frames = load_final_qc_pair({'vanilla': config['vanilla'], 'masked': config['masked']})
    for region, aliases in REGIONS[config['channel']].items():
        ranking, figure = plot_sampling_depth_condition(
            repo_root, config['condition'], config['channel'], region,
            frames, aliases, config['colors'], top_n=8,
        )
        if figure is None:
            print(f'No complete shared columns: {name} | {region}')
            continue
        display(figure)
        safe_name = name.lower().replace(' ', '_').replace('.', '')
        safe_region = region.replace(' ', '_')
        figure.savefig(figure_dir / f'{safe_name}_{safe_region}.png', dpi=400, bbox_inches='tight')
        ranking.to_csv(table_dir / f'{safe_name}_{safe_region}_ranking.csv', index=False)
        ranking_tables[(name, region)] = ranking
        summary_rows.extend(summarize_sampling_depth(name, region, frames, ranking))
        plt.close(figure)

sampling_summary = pd.DataFrame(summary_rows)
sampling_summary.to_csv(table_dir / 'sampling_depth_summary.csv', index=False)
display(sampling_summary.round(3))


## Cross-condition sensitivity summary

Values close between the two bars indicate that the qualitative masking effect is already visible with QC survivors from nominal first 100. Large disagreements identify conclusions that depend on deeper sampling and should not be inferred from a small ensemble.


In [ ]:
plot_data = sampling_summary.copy()
plot_data['Condition and region'] = plot_data['Condition'] + ' | ' + plot_data['Region']
subset_palette = {'Complete final QC': ACCENT_PALETTE['PINK'], 'QC survivors from nominal first 100': ACCENT_PALETTE['CORAL']}
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
sns.barplot(data=plot_data, x='Condition and region', y='Median masked/vanilla IQR ratio', hue='Subset', palette=subset_palette, ax=axes[0])
axes[0].axhline(1, color='#555555', linewidth=0.8, linestyle='--')
axes[0].set_title('Magnitude of the masking-associated variability difference')
sns.barplot(data=plot_data, x='Condition and region', y='Fraction broader under masking', hue='Subset', palette=subset_palette, ax=axes[1])
axes[1].axhline(0.5, color='#555555', linewidth=0.8, linestyle='--')
axes[1].set_title('Fraction of coordinates broader in masked ensembles')
axes[1].tick_params(axis='x', rotation=55)
for axis in axes:
    axis.grid(axis='x', visible=False)
    sns.despine(ax=axis)
fig.suptitle('Sampling-depth sensitivity across final-QC ensembles', fontsize=17, fontweight='semibold')
fig.tight_layout()
display(fig)
fig.savefig(figure_dir / 'sampling_depth_cross_condition_summary.png', dpi=400, bbox_inches='tight')


## Cross-project retention by recycle

These heatmaps use the same final-QC CSVs analyzed above. Each cell corresponds to one recycle and is annotated with the number of nominal seed/model trajectories represented after final QC. The upper row uses the fixed first 20 seeds (100 nominal trajectories); the lower row uses all 100 seeds (500 nominal trajectories). Missing trajectories are not replaced. Color gradients follow the project palettes: blue for CaV1.2, green for Kv2.1, and purple for NaV1.5.

In [ ]:
from shared.recycle_retention_analysis import run_recycle_retention_analysis
recycle_retention_table, recycle_retention_figures = run_recycle_retention_analysis(
    DATASETS,
    repo_root/'docs'/'figures'/'sampling_depth',
    repo_root/'docs'/'tables'/'sampling_depth',
)
display(recycle_retention_table)
display(recycle_retention_figures['retained_fraction'])
display(recycle_retention_figures['excluded'])
plt.close('all')

## Cross-project retention by recycle

These heatmaps use the same final-QC CSVs analyzed above. Each cell corresponds to one recycle and is annotated with the number of nominal seed/model trajectories represented after final QC. The upper row uses the fixed first 20 seeds (100 nominal trajectories); the lower row uses all 100 seeds (500 nominal trajectories). Missing trajectories are not replaced. Color gradients follow the project palettes: blue for CaV1.2, green for Kv2.1, and purple for NaV1.5.

In [ ]:
from shared.recycle_retention_analysis import run_recycle_retention_analysis
recycle_retention_table, recycle_retention_figures = run_recycle_retention_analysis(
    DATASETS,
    repo_root/'docs'/'figures'/'sampling_depth',
    repo_root/'docs'/'tables'/'sampling_depth',
)
display(recycle_retention_table)
display(recycle_retention_figures['retained_fraction'])
display(recycle_retention_figures['excluded'])
plt.close('all')

## Cross-project retention by recycle

These heatmaps use the same final-QC CSVs analyzed above. Each cell corresponds to one recycle and is annotated with the number of nominal seed/model trajectories represented after final QC. The upper row uses the fixed first 20 seeds (100 nominal trajectories); the lower row uses all 100 seeds (500 nominal trajectories). Missing trajectories are not replaced. Color gradients follow the project palettes: blue for CaV1.2, green for Kv2.1, and purple for NaV1.5.

In [ ]:
from shared.recycle_retention_analysis import run_recycle_retention_analysis
recycle_retention_table, recycle_retention_figures = run_recycle_retention_analysis(
    DATASETS,
    repo_root/'docs'/'figures'/'sampling_depth',
    repo_root/'docs'/'tables'/'sampling_depth',
)
display(recycle_retention_table)
display(recycle_retention_figures['retained_fraction'])
display(recycle_retention_figures['excluded'])
plt.close('all')

## L403A all-distance equal-seed-depth extension

The panels above test selected structural coordinates using the same fixed nominal early cohort: the first 20 ordered seeds × five model trajectories = 100 intended trajectories per protocol, followed by the final-QC cross-check. The raw Kv2.1 tables share 1,410 distance columns, but 864 are raw interchain A–B/A–C/etc. measurements that are not invariant to homotetramer chain relabeling. Those columns are excluded from broadening inference. This extension analyzes the 546 chain-label-safe intrachain distances (273 Cα and 273 shortest-heavy-atom distances).

At each depth, the nominal seed/model cohort is fixed before QC and missing trajectories are not replaced. One structure represents each surviving trajectory: the latest recycle that passes final QC. The nominal generated-trajectory budgets are 20, 40, 60, 80, and 100 (4, 8, 12, 16, and 20 seeds), followed by the complete 100-seed cohort. This is deterministic rather than timestamped chronological generation. Global IQR is descriptive. Primary per-distance inference uses one within-seed IQR and compares vanilla versus masked seed distributions with two-sided Mann–Whitney tests followed by Benjamini–Hochberg FDR correction within each depth. Brown–Forsythe tests provide a trajectory-level sensitivity analysis. Wasserstein separation uses the single representatives.

In [ ]:
from IPython.display import Image, display
from kv21.run_l403a_all_distance_sampling_analysis import run as run_all_distance_sampling, TAB as ALL_DISTANCE_TAB, FIG as ALL_DISTANCE_FIG
all_distance_result = run_all_distance_sampling()
paper_statistics = pd.read_csv(ALL_DISTANCE_TAB/'l403a_all_distance_paper_statistics_summary.csv')
stability_statistics = pd.read_csv(ALL_DISTANCE_TAB/'l403a_all_distance_sampling_stability.csv')
trajectory_retention_summary = pd.read_csv(ALL_DISTANCE_TAB/'l403a_first100_nominal_trajectory_qc_summary.csv')
trajectory_retention_audit = pd.read_csv(ALL_DISTANCE_TAB/'l403a_first100_nominal_trajectory_qc_audit.csv')
retention_test = pd.read_csv(ALL_DISTANCE_TAB/'l403a_first100_qc_retention_fisher_test.csv')
overall_seed_breadth_summary = pd.read_csv(ALL_DISTANCE_TAB/'l403a_first100_seed_level_global_breadth_summary.csv')
equal_count_rarefaction_summary = pd.read_csv(ALL_DISTANCE_TAB/'l403a_first100_equal_count_rarefaction_summary.csv')
random_seed_saturation_summary = pd.read_csv(ALL_DISTANCE_TAB/'l403a_random_seed_saturation_summary.csv')
first100_distribution_statistics_summary = pd.read_csv(ALL_DISTANCE_TAB/'l403a_first100_seed_block_distribution_statistics_summary.csv')
full_distribution_statistics_summary = pd.read_csv(ALL_DISTANCE_TAB/'l403a_full_seed_block_distribution_statistics_summary.csv')
first100_vs_full_directional_statistics = pd.read_csv(ALL_DISTANCE_TAB/'l403a_first100_vs_full_directional_breadth_statistics.csv')
display(paper_statistics)
display(stability_statistics)

### Final-QC cross-check for the nominal first 100 trajectories

In [ ]:
display(trajectory_retention_summary)
display(retention_test)
display(trajectory_retention_audit.loc[~trajectory_retention_audit['final_qc_retained']])
display(Image(filename=str(ALL_DISTANCE_FIG/'first100_nominal_trajectory_qc_audit.png')))

### Overall breadth and equal-retained-count statistical checks

In [ ]:
display(overall_seed_breadth_summary)
display(equal_count_rarefaction_summary)
display(Image(filename=str(ALL_DISTANCE_FIG/'first100_bang_for_buck_breadth_statistics.png')))

### First-100 distribution statistics using the standard W1 framework

In [ ]:
display(first100_distribution_statistics_summary)
display(Image(filename=str(ALL_DISTANCE_FIG/'first100_seed_block_distribution_statistics.png')))

### First-100 headline figures: breadth gained per fixed sampling budget

The four-panel summary separates the main claims: one breadth value per independent seed; the distribution of breadth effects across all 546 chain-label-safe intrachain distances; the direct one-sided seed-block permutation result after BH-FDR; and final-QC retention from the same 100 nominal trajectories. The compact evidence map orders all distances from narrower to broader and marks those with direct statistical support for masked broadening.

In [ ]:
display(Image(filename=str(ALL_DISTANCE_FIG/'first100_masked_sampling_breadth_main_summary.png')))
display(Image(filename=str(ALL_DISTANCE_FIG/'first100_retention_vs_breadth_heatmap_scorecard.png')))
display(Image(filename=str(ALL_DISTANCE_FIG/'first100_all_distance_breadth_evidence_map.png')))

### Direct statistical test of broader distributions: first 100 versus full QC

For each chain-label-safe intrachain distance, the directional null is tested by permuting complete seed blocks between protocols. The one-sided alternative is `IQR(masked) > IQR(vanilla)`. Benjamini–Hochberg correction is applied across all 546 distances separately within each cohort. This is the direct broadening test; W1 above tests any distributional separation and is not itself directional.

In [ ]:
display(pd.concat([
    first100_distribution_statistics_summary.assign(cohort='Nominal first 100'),
    full_distribution_statistics_summary.assign(cohort='Full QC')
], ignore_index=True))
display(first100_vs_full_directional_statistics.sort_values(
    ['q_broader_masked_seed_block_BH_first100','q_broader_masked_seed_block_BH_full']
).head(30))
display(Image(filename=str(ALL_DISTANCE_FIG/'first100_vs_full_breadth_summary_heatmaps.png')))
display(Image(filename=str(ALL_DISTANCE_FIG/'first100_vs_full_directional_breadth_heatmap.png')))

### Clear reproducibility views

The first heatmap counts distances according to whether the direct masked-broadening test is significant in the first-100 and full-QC cohorts. The scatter then shows the effect magnitude for every distance: upper-right points are broader under masking in both analyses, and points near the diagonal have similar early and full effects. The final heatmaps separate Cα and shortest-heavy distances to show whether the conclusion depends on the distance definition.

In [ ]:
display(pd.read_csv(ALL_DISTANCE_TAB/'l403a_first100_vs_full_broadening_concordance.csv', index_col=0))
display(Image(filename=str(ALL_DISTANCE_FIG/'first100_vs_full_broadening_concordance_heatmap.png')))
display(Image(filename=str(ALL_DISTANCE_FIG/'first100_vs_full_breadth_quadrant_scatter.png')))
display(pd.read_csv(ALL_DISTANCE_TAB/'l403a_first100_vs_full_breadth_by_distance_type.csv'))
display(Image(filename=str(ALL_DISTANCE_FIG/'first100_vs_full_breadth_by_distance_type_heatmaps.png')))

### Random-seed sampling-efficiency and saturation curves

In [ ]:
display(random_seed_saturation_summary)
display(Image(filename=str(ALL_DISTANCE_FIG/'random_seed_sampling_efficiency_saturation.png')))

### How widespread is masked broadening across sampling depth?

In [ ]:
display(Image(filename=str(ALL_DISTANCE_FIG/'all_distance_breadth_fraction_by_depth.png')))

### QC survivors from the nominal first 100 versus full-QC breadth

In [ ]:
display(Image(filename=str(ALL_DISTANCE_FIG/'first100_nominal_vs_full_distance_breadth.png')))

### Ensemble-wide distribution of masked/vanilla IQR ratios

In [ ]:
display(Image(filename=str(ALL_DISTANCE_FIG/'all_distance_iqr_ratio_distributions.png')))

### Strongest reproducible broadened and narrowed contacts

In [ ]:
display(pd.read_csv(ALL_DISTANCE_TAB/'top_concordant_distance_statistical_details.csv'))
display(Image(filename=str(ALL_DISTANCE_FIG/'top_all_distance_breadth_heatmap.png')))

### Representative chain-label-safe intrachain distance distributions

In [ ]:
display(Image(filename=str(ALL_DISTANCE_FIG/'representative_all_distance_distributions.png')))

### Statistical quantities used in this extension

- **Effect size:** masked/vanilla global IQR ratio and log2 ratio for each distance.
- **Chain safety:** raw Kv2.1 interchain columns are excluded; inference uses 546 same-chain distances that are invariant to subunit relabeling.
- **Trajectory representation:** latest final-QC recycle, one structure per surviving seed/model trajectory.
- **Cluster-aware breadth:** median within-seed IQR and masked/vanilla seed-IQR ratio.
- **Primary per-distance inference:** two-sided Mann–Whitney comparison of seed IQRs with within-depth BH-FDR (`seed_breadth_q`).
- **Overall breadth test:** one median normalized all-distance IQR per seed, compared as 20 vanilla versus 20 masked seeds.
- **Standard distribution test:** W1 and normalized W1 per distance, with 999 unpaired seed-block permutations, 500 seed-block bootstrap replicates, and BH-FDR across the 546 intrachain distances.
- **Sensitivity analyses:** trajectory-level Brown–Forsythe tests and repeated vanilla rarefaction to 85 retained trajectories.
- **Sampling saturation:** 1,000 random seed subsets at 5, 10, 20, 25, 50, 75, and 100 input seeds.
- **Distributional separation:** Wasserstein distance between one representative per retained trajectory, in Å and normalized by pooled IQR.
- **Sampling-depth reproducibility:** Spearman correlation and sign agreement between QC survivors from the nominal first-100 cohort (20 seeds × five models) and full-QC distance effects.

Very large ratios can result when the vanilla IQR is nearly zero. Individual highlighted contacts should therefore be reported with both raw IQRs and Wasserstein effect sizes, not ratios alone.